In [1]:
import time
import glob
import numpy as np
import torch.nn as nn

from PIL import Image
from multiprocessing import cpu_count
from datasets import load_dataset
from torch.nn import NLLLoss
from torch.optim import Adam
from torch.utils.data import DataLoader

from geqie_qml import VQCLayer, MatrixDataset, compute_and_save_circuits

def train_QCNN(
    epochs: int = 20,
    num_classes: int = 2,
    batch_size: int = 5,
    num_qubits: int = 9,
    num_layers: int = 1,
    circuits_directory: str = "circuits",
    num_workers: int | None = None,
):
    """
    Train a VQCLayer followed by a classical linear head.
 
    The model is assembled here to illustrate the intended compositional
    pattern — VQCLayer produces probability features, the linear head maps
    them to class logits, and LogSoftmax + NLLLoss combine as cross-entropy.
 
    Parameters
    ----------
    epochs, num_classes, batch_size : int
    num_qubits : int
        Must match the unitary matrices stored in circuits_directory.
    num_layers : int
        Number of brickwork VQC layers.
    circuits_directory : str
        Directory containing .npz files produced by compute_and_save_circuits.
    num_workers : int | None
        Worker processes for parallel simulation.  Defaults to cpu_count - 1.
    """
    # --- Build model from composable parts ---
    vqc_layer = VQCLayer(num_qubits=num_qubits, num_layers=num_layers)
    model = nn.Sequential(
        vqc_layer,
        nn.Linear(2 ** num_qubits, num_classes),
        nn.LogSoftmax(dim=-1),
    )
 
    optimizer = Adam([
        {"params": vqc_layer.quantum_weight,          "lr": 0.001},
        {"params": model[1].parameters(),             "lr": 0.01},
    ])
 
    f_loss = NLLLoss()
    model.train()
    loss_list = []
 
    files = sorted(glob.glob(f"{circuits_directory}/*.npz"))
    print(f"Found {len(files)} circuit files")
    dataset = MatrixDataset(files)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
 
    if num_workers is None:
        num_workers = max(1, cpu_count() - 1)
    print(f"Starting training with {num_workers} worker processes")
 
    # The parallel_context() block starts and owns the process pool.
    # No manual executor injection needed — vqc_layer handles it internally.
    with vqc_layer.parallel_context(num_workers=num_workers):
        for epoch in range(epochs):
            start = time.time()
            total_loss = []
 
            for batch_matrices, batch_labels in dataloader:
                optimizer.zero_grad()
 
                output = model(batch_matrices)
                loss = f_loss(output, batch_labels)
                loss.backward()
 
                total_loss.append(loss.item())
                optimizer.step()
 
            epoch_loss = sum(total_loss) / len(total_loss)
            loss_list.append(epoch_loss)
            elapsed = time.time() - start
            print(
                f"Epoch {epoch + 1}/{epochs}  "
                f"Loss: {epoch_loss:.4f}  "
                f"Time: {elapsed:.2f}s"
            )
 
    return loss_list, model

c:\Python\QML\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_and_process_mnist_dataset(
    path_to_mnist_dataset,
    labels_to_include=[0, 1],
    n_samples_per_label=100,
    resize=(8, 8),
):
    mnist_dataset = load_dataset("parquet", data_files=path_to_mnist_dataset)
 
    selected_images = {x: [] for x in labels_to_include}
    for image_idx in range(len(mnist_dataset["train"])):
        label = mnist_dataset["train"][image_idx]["label"]
        if label in labels_to_include:
            if len(selected_images[label]) < n_samples_per_label:
                resized_image = mnist_dataset["train"][image_idx]["image"].resize(
                    resize, resample=Image.BILINEAR
                )
                selected_images[label].append({
                    "image": np.array(resized_image),
                    "label": label
                })
 
    selected_images = [item for sublist in selected_images.values() for item in sublist]
 
    X = np.array([item["image"] for item in selected_images])
    y = np.array([item["label"] for item in selected_images])
    return X, y

In [ ]:
path_to_mnist_dataset = ''
circuits_directory = ''

In [ ]:
X, y = load_and_process_mnist_dataset(
    path_to_mnist_dataset,
    labels_to_include=[0, 1],
    n_samples_per_label=10,
    resize=(16, 16),
)

print("Running")
compute_and_save_circuits(X, y, save_dir=circuits_directory, number_of_workers=4)
print("Circuits computed")
loss_list, model = train_QCNN(epochs=10, num_classes=2, circuits_directory=circuits_directory)

Running
Circuits computed
Found 20 circuit files
Starting training with 15 worker processes
Epoch 1/10  Loss: 2.1556  Time: 3.66s
Epoch 2/10  Loss: 0.1261  Time: 1.04s
Epoch 3/10  Loss: 0.2328  Time: 1.02s
Epoch 4/10  Loss: 0.0006  Time: 1.15s
Epoch 5/10  Loss: 0.0001  Time: 1.15s
Epoch 6/10  Loss: 0.0000  Time: 1.15s
Epoch 7/10  Loss: 0.0000  Time: 1.16s
Epoch 8/10  Loss: 0.0000  Time: 1.16s
Epoch 9/10  Loss: 0.0000  Time: 1.08s
Epoch 10/10  Loss: 0.0001  Time: 1.05s


In [7]:
loss_list

[2.155611079186201,
 0.12613349170351285,
 0.23278823268071847,
 0.0006233587584247857,
 0.00011909594707049109,
 6.33585948861537e-06,
 9.953926376482514e-07,
 8.999514601981673e-06,
 1.0864785929598497e-05,
 0.00013815760606217253]

In [8]:
model

Sequential(
  (0): VQCLayer(num_qubits=9, num_layers=1, shots=1024, scale_output=True, num_params=27)
  (1): Linear(in_features=512, out_features=2, bias=True)
  (2): LogSoftmax(dim=-1)
)